In [0]:
employee_df = spark.read.csv("/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/employee.csv",header=True,inferSchema=True,sep="|",quote="'")
employee_df.display()

##Handling Missing Record

In [0]:
employee_df.na.drop(how='any').display()
employee_df.na.drop(how='all').display()
employee_df.na.drop(subset=['id','name']).display()

In [0]:
from pyspark.sql.functions import col
employee_df.filter(col("id").isNull()).display()

In [0]:
employee_df.filter(~(col('id').isNull()) & ~col('name').isNull()).display()

In [0]:
employee_df.na.fill("NULL in source",subset=["name","gen"]).na.fill(-1,subset=["id"]).na.fill(0,subset=["exp"]).display()

In [0]:
missing_values={
    "name":"ANONYMOUS",
    "gen":"UNKNOWN",
    "id":-1,
    "exp":0,
}
employee_df.na.fill(missing_values).display()

In [0]:
from pyspark.sql.functions import when 
employee_df.withColumn("name",when(col("name").isNull(),"ANONYMOUS").otherwise(col("name"))).display()

In [0]:
employee_df.withColumn("exp",when(col("exp").isNull(),0).otherwise(col("exp"))).display()

In [0]:
from pyspark.sql import functions as f

In [0]:
employee_df.na.fill((f.current_date()),subset=['doj']).display()

In [0]:
employee_df.withColumn('doj', when(col('doj').isNull(), f.current_date()).otherwise(col('doj'))).display()

In [0]:
from pyspark.sql.functions import avg 
avg_exp = employee_df.select(avg("exp")).collect()[0][0]
avg_exp

In [0]:
from pyspark.sql.functions import current_date
missing_values={
    "name":"ANONYMOUS",
    "gen":"UNKNOWN",
    "id":-1,
    "exp":round(avg_exp),
}
employee_df.na.fill(missing_values).display()

In [0]:
employee_df.withColumn(
    "gen",
    when(col("gen") == "M", "Male")
    .when(col("gen") == "F", "Female")
    .when(col("gen") == "T", "Transgender")
    .otherwise("Invalid")
).display()

In [0]:
from pyspark.sql.functions import avg, current_date

avg_exp = employee_df.select(avg("exp")).collect()[0][0]
mode_gender = (
    employee_df.groupBy("gen").count().sort(col("count"), ascending=False).first()[0][0]
)

MISSING_DEFAULT_VALUES = {
    "name": "Anonymous",
    "id": -1,
    "exp": round(avg_exp),
    "gen": mode_gender
}

employee_df.na.fill(MISSING_DEFAULT_VALUES).display()

In [0]:
employee_df.na.replace({
    "M":"Male",
    "F":"Female",
    "T":"Transgender"
},subset=["gen"]).display()